# 01. 데이터 수집 — 전국 아파트 실거래가

**데이터 출처**: 국토교통부 아파트매매 실거래상세자료 (공공데이터포털 Open API)
**이 노트북에서 하는 일**: API로 아파트 실거래 데이터를 수집해서 통합하고, 다음 노트북(`02_eda_feature_engineering`)이 바로 이어받을 수 있도록 `data/processed/` 아래에 저장합니다.

> 노트북을 5단계(01 데이터 → 02 EDA/피처엔지니어링 → 03 회귀 → 04 군집 → 05 분류)로 나눴습니다. 각 노트북은 이전 단계 산출물을 **파일로 저장 → 다음 단계에서 파일로 로드**하는 방식이라, 커널을 껐다 켜도 이전 노트북부터 순서대로 다시 실행할 필요 없이 해당 노트북만 열어서 이어가면 됩니다.

## 1. 사전 준비 — API 인증키 발급 방법

### 1) 공공데이터포털 회원가입
- https://www.data.go.kr 접속 → 우측 상단 **회원가입** (개인 회원으로 충분)

### 2) API 검색 및 활용신청
- 로그인 후 검색창에 **"아파트 매매 실거래가 상세 자료"** 검색
- **정확한 데이터셋명: `국토교통부_아파트 매매 실거래가 상세 자료` (data.go.kr ID: 15126468, 엔드포인트: `RTMSDataSvcAptTradeDev`)**
  - 이름이 비슷한 `국토교통부_아파트 매매 실거래가 자료` (ID: 15126469, "상세"가 빠진 이름, 엔드포인트: `RTMSDataSvcAptTrade`)는 **필드 수가 더 적은 구버전**이므로 헷갈리지 않도록 주의. 이 노트북은 **"상세 자료"(15126468, `...Dev`)** 기준.
- 상세 페이지에서 **활용신청** 클릭 후 간단한 정보 입력 (심의 없이 자동 승인되는 경우가 대부분)

### 3) 승인 대기 및 키 확인
- 보통 1~2시간 이내 자동 승인
- 승인 후 **마이페이지 → 데이터활용 → 개발계정**에서 서비스키 확인
- **Decoding 키**를 사용 (Encoding 키를 쓰면 이중 인코딩되어 403 에러 발생)

### 4) 트래픽 한도
- 개발계정은 보통 일일 트래픽 한도가 있어서(통상 1,000회 내외), 전국 267개 시군구 × 여러 개월을 수집하려면 한도를 초과할 수 있음 → 아래 수집 코드는 **지역/월 단위 체크포인트 저장**으로 설계 (중단 후 재실행하면 이어서 수집)

### 5) 키 보관 — `.env` 파일 사용
프로젝트 루트에 `.env` 파일을 만들고 아래처럼 작성 (절대 커밋하지 말 것, `.gitignore`에 `.env` 추가):
```
DATA_GO_KR_API_KEY=발급받은_디코딩_키
```


In [ ]:
import os
import time
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


# ------------------------------------------------------------------
# 설정
# ------------------------------------------------------------------

DEMO_MODE = False

load_dotenv()
SERVICE_KEY = os.environ.get(
    "DATA_GO_KR_API_KEY",
    ""
)


# ------------------------------------------------------------------
# 경로 설정
# ------------------------------------------------------------------

# Apartment price/notebook/v1_1year_notebook
PROJECT_DIR = Path.cwd().parents[1]

DATA_ROOT = PROJECT_DIR / "data"

# 기존 전국 1년 데이터
DATA_DIR = DATA_ROOT / "v1_1year"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# V1 / V2 공통
EXTERNAL_DIR = DATA_ROOT / "external"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXTERNAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------------
# API 키 확인
# ------------------------------------------------------------------

if not DEMO_MODE and not SERVICE_KEY:
    raise ValueError(
        "DEMO_MODE=False 인데 DATA_GO_KR_API_KEY "
        "환경변수가 비어 있습니다. "
        "서비스키를 환경변수로 설정하거나 "
        "DEMO_MODE=True 로 두세요."
    )


# ------------------------------------------------------------------
# 확인
# ------------------------------------------------------------------

print(f"DEMO_MODE = {DEMO_MODE}")

print("\n[경로 확인]")
print("현재 위치       :", Path.cwd())
print("프로젝트 루트   :", PROJECT_DIR.resolve())
print("DATA_ROOT       :", DATA_ROOT.resolve())
print("DATA_DIR        :", DATA_DIR.resolve())
print("RAW_DIR         :", RAW_DIR.resolve())
print("PROCESSED_DIR   :", PROCESSED_DIR.resolve())
print("EXTERNAL_DIR    :", EXTERNAL_DIR.resolve())

DEMO_MODE = False


## 2. 지역코드(법정동코드) 조회

Open API는 지역을 5자리 **법정동시군구코드(LAWD_CD)**로 지정합니다.
`PublicDataReader`가 제공하는 `code_bdong()`으로 전국 시군구 코드를 한 번에 받아올 수 있습니다 (별도 인증키 불필요).

In [18]:
import PublicDataReader as pdr

def get_nationwide_sigungu_codes() -> pd.DataFrame:
    """전국 시군구 단위 법정동코드 목록을 반환한다 (읍면동 레벨 제외, 폐지되지 않은 코드만)."""
    code_df = pdr.code_bdong()
    sgg = code_df[
        (code_df["읍면동명"] == "") &
        (code_df["시군구명"] != "") &
        (code_df["말소일자"] == "")
    ][["시도명", "시군구명", "시군구코드"]].drop_duplicates().reset_index(drop=True)
    return sgg

sigungu_df = get_nationwide_sigungu_codes()
print(f"전국 시군구 수: {len(sigungu_df)}")

# 다음 단계(EDA)에서 시군구명을 매핑할 때 쓸 수 있게 저장
sigungu_df.to_csv(PROCESSED_DIR / "sigungu_map.csv", index=False, encoding="utf-8-sig")
sigungu_df.head(10)


출처: 행정기관(행정동) 및 관할구역(법정동) 변경내역(2024.8.1. 시행)
전국 시군구 수: 267


,시도명,시군구명,시군구코드
0,서울특별시,종로구,11110
1,서울특별시,중구,11140
2,서울특별시,용산구,11170
3,서울특별시,성동구,11200
4,서울특별시,광진구,11215
5,서울특별시,동대문구,11230
6,서울특별시,중랑구,11260
7,서울특별시,성북구,11290
8,서울특별시,강북구,11305
9,서울특별시,도봉구,11320


## 3. 수집 대상(지역/기간) 설정

- 전국 전체를 한 번에 수집하면 호출 수가 많아(시군구 267개 × 개월 수) 시간이 오래 걸리고 개발계정 일일 트래픽 한도에 걸릴 수 있음
- `TEST_SUBSET=True`면 서울 5개 구만 우선 수집해 파이프라인을 검증하고, 문제 없으면 `TEST_SUBSET=False`로 바꿔 전국 전체를 수집

In [19]:
TEST_SUBSET = False   # 처음엔 True로 소규모 검증 후, 전국 수집 시 False로 변경

if TEST_SUBSET:
    target_regions = sigungu_df[sigungu_df["시도명"] == "서울특별시"].head(5)
else:
    target_regions = sigungu_df.copy()

# 수집할 계약월 범위 (YYYYMM), 최근 12개월 예시 -> 필요에 맞게 조정
END_MONTH = pd.Timestamp.today().to_period("M")
START_MONTH = END_MONTH - 11  # 최근 12개월
year_months = [str(p) for p in pd.period_range(START_MONTH, END_MONTH, freq="M")]
year_months = [ym.replace("-", "") for ym in year_months]

print(f"대상 지역 수: {len(target_regions)}, 대상 월 수: {len(year_months)}")
print(f"예상 API 호출 수(지역 x 월): {len(target_regions) * len(year_months)}")


대상 지역 수: 267, 대상 월 수: 12
예상 API 호출 수(지역 x 월): 3204


## 4. 데이터 수집 함수

- `fetch_real(...)`: `PublicDataReader.TransactionPrice`로 실제 API를 호출
- `fetch_demo(...)`: 서비스키 없이 파이프라인을 검증할 수 있도록 실제 응답과 동일한 컬럼 구조의 합성 데이터 생성
- 지역x월 조합마다 결과를 `data/raw/{시군구코드}_{연월}.csv`로 저장 (중단되어도 이어서 수집 가능한 체크포인트 방식)

In [21]:
from PublicDataReader import TransactionPrice

if not DEMO_MODE:
    api = TransactionPrice(SERVICE_KEY)

def fetch_real(sigungu_code: str, year_month: str) -> pd.DataFrame:
    return api.get_data(
        property_type="아파트",
        trade_type="매매",
        sigungu_code=sigungu_code,
        year_month=year_month,
    )

def fetch_demo(sigungu_code: str, year_month: str, n: int = 30, seed: int | None = None) -> pd.DataFrame:
    """실제 API 응답과 동일한 주요 컬럼을 갖는 합성 데이터를 생성 (구조 검증용)."""
    rng = np.random.default_rng(seed or (hash((sigungu_code, year_month)) % (2**32)))
    n = rng.integers(5, n)
    base_price = rng.uniform(40000, 150000)  # 만원 단위
    df = pd.DataFrame({
        "법정동시군구코드": sigungu_code,
        "계약년도": int(year_month[:4]),
        "계약월": int(year_month[4:6]),
        "계약일": rng.integers(1, 28, n),
        "전용면적": np.round(rng.uniform(39, 135, n), 2),
        "층": rng.integers(1, 25, n),
        "건축년도": rng.integers(1988, 2024, n),
        "거래금액": np.round(base_price + rng.normal(0, 8000, n)).clip(min=5000),
        "아파트": [f"데모아파트{i}" for i in rng.integers(1, 20, n)],
        "법정동": "데모동",
    })
    return df

def collect_transaction_data(regions: pd.DataFrame, months: list[str], sleep_sec: float = 0.2) -> None:
    """지역x월 조합을 순회하며 체크포인트 파일로 저장한다. 이미 저장된 파일은 건너뛴다."""
    total = len(regions) * len(months)
    done = 0
    for _, row in regions.iterrows():
        sgg_code = row["시군구코드"]
        for ym in months:
            out_path = RAW_DIR / f"{sgg_code}_{ym}.csv"
            done += 1
            if out_path.exists():
                continue
            try:
                if DEMO_MODE:
                    df = fetch_demo(sgg_code, ym)
                else:
                    df = fetch_real(sgg_code, ym)
                    time.sleep(sleep_sec)  # 트래픽 한도 보호를 위한 딜레이
                df.to_csv(out_path, index=False, encoding="utf-8-sig")
            except Exception as e:
                print(f"[실패] {row['시도명']} {row['시군구명']} {ym}: {e}")
            if done % 50 == 0 or done == total:
                print(f"진행: {done}/{total}")

collect_transaction_data(target_regions, year_months)


In [22]:
print(len(year_months))
print(year_months[:3], "...", year_months[-3:])

12
['202509', '202510', '202511'] ... ['202606', '202607', '202608']


In [23]:
import glob

raw_files = glob.glob(str(RAW_DIR / "*.csv"))
print(f"파일 수: {len(raw_files)}")

sample_df = pd.read_csv(raw_files[0], encoding="utf-8-sig")
print(raw_files[0])
print(sample_df[["법정동", "계약년도", "계약월"]].head() if "법정동" in sample_df.columns else sample_df.head())

파일 수: 3204
..\data\raw\11110_202509.csv
   법정동  계약년도  계약월
0  숭인동  2025    9
1  창신동  2025    9
2  사직동  2025    9
3  교북동  2025    9
4  사직동  2025    9


## 5. 수집 데이터 통합 및 저장

체크포인트 CSV들을 하나로 합치고, 다음 노트북(`02_eda_feature_engineering.ipynb`)이 바로 로드할 수 있도록 parquet으로 저장합니다.

object(문자열) 컬럼을 저장 전에 명시적으로 `str`로 캐스팅합니다 — 안 하면 `지번` 같은 컬럼에 문자열/숫자가 섞여 있을 때 `to_parquet()`에서 `ArrowTypeError`가 납니다.

In [24]:
import glob

raw_files = glob.glob(str(RAW_DIR / "*.csv"))
print(f"체크포인트 파일 수: {len(raw_files)}")

df_list = [pd.read_csv(f, encoding="utf-8-sig") for f in raw_files]
raw_df = pd.concat(df_list, ignore_index=True)
print(raw_df.shape)

체크포인트 파일 수: 3204
(474066, 64)


In [25]:
# object 컬럼 타입 통일 후 저장 (혼합 타입으로 인한 parquet 저장 에러 방지)
object_cols = raw_df.select_dtypes(include="object").columns
for col in object_cols:
    raw_df[col] = raw_df[col].astype(str)

raw_df.to_parquet(PROCESSED_DIR / "apt_trade_raw.parquet", index=False)
print(f"저장 완료: {PROCESSED_DIR / 'apt_trade_raw.parquet'} ({raw_df.shape[0]}행)")
raw_df.head()


저장 완료: ..\data\processed\apt_trade_raw.parquet (474066행)


,법정동시군구코드,법정동읍면동코드,법정동지번코드,법정동본번코드,법정동부번코드,도로명,도로명시군구코드,도로명코드,도로명일련번호코드,도로명지상지하코드,...,aptSeq,cdealType,cdealDay,dealingGbn,estateAgentSggNm,rgstDate,aptDong,slerGbn,buyerGbn,landLeaseholdGbn
0,11110.0,17500.0,1.0,766.0,0.0,숭인동길,11110.0,4100204.0,1.0,0.0,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,11110.0,17400.0,1.0,703.0,0.0,낙산길,11110.0,4100020.0,2.0,0.0,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,11110.0,11500.0,1.0,9.0,0.0,사직로8길,11110.0,4100135.0,3.0,0.0,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
3,11110.0,18000.0,1.0,126.0,0.0,송월길,11110.0,4100192.0,1.0,NaN,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
4,11110.0,11500.0,1.0,9.0,0.0,사직로8길,11110.0,4100135.0,3.0,0.0,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [26]:
print(TEST_SUBSET)
print(target_regions[["시도명", "시군구명"]].drop_duplicates())

False
         시도명 시군구명
0      서울특별시  종로구
1      서울특별시   중구
2      서울특별시  용산구
3      서울특별시  성동구
4      서울특별시  광진구
..       ...  ...
262  전북특별자치도  장수군
263  전북특별자치도  임실군
264  전북특별자치도  순창군
265  전북특별자치도  고창군
266  전북특별자치도  부안군

[267 rows x 2 columns]


In [27]:
import glob
raw_files = glob.glob(str(RAW_DIR / "*.csv"))
print(f"저장된 체크포인트 파일 수: {len(raw_files)}")

# 파일명에서 지역코드만 추출해서 어떤 지역들이 실제로 수집됐는지 확인
region_codes_in_files = set(f.split("\\")[-1].split("_")[0] for f in raw_files)  # Windows 경로 기준
print(f"실제 수집된 지역 수: {len(region_codes_in_files)}")

저장된 체크포인트 파일 수: 3204
실제 수집된 지역 수: 267


In [28]:
import glob

raw_files = glob.glob(str(RAW_DIR / "*.csv"))
print(f"체크포인트 파일 수: {len(raw_files)}")

df_list = [pd.read_csv(f, encoding="utf-8-sig") for f in raw_files]
raw_df = pd.concat(df_list, ignore_index=True)
print(raw_df.shape)

체크포인트 파일 수: 3204
(474066, 64)


In [29]:
print(target_regions["시도명"].unique())
print(len(target_regions))

['서울특별시' '부산광역시' '대구광역시' '인천광역시' '광주광역시' '대전광역시' '울산광역시' '경기도' '충청북도'
 '충청남도' '전라남도' '경상북도' '경상남도' '제주특별자치도' '강원특별자치도' '전북특별자치도']
267


In [30]:
collect_transaction_data(target_regions, year_months)

In [32]:
raw_df.describe()

,법정동시군구코드,법정동읍면동코드,법정동지번코드,법정동본번코드,법정동부번코드,도로명시군구코드,도로명코드,도로명일련번호코드,도로명지상지하코드,도로명건물본번호코드,도로명건물부번호코드,전용면적,계약년도,계약월,계약일,거래금액,층,건축년도
count,474066.000000,474066.000000,474066.000000,474066.000000,474066.000000,473380.000000,4.733530e+05,473361.000000,430429.0,473274.000000,473274.000000,474066.000000,474066.000000,474066.000000,474066.000000,4.740660e+05,474066.000000,474066.000000
mean,35338.463663,13226.944927,1.011642,826.856400,3.293725,35351.183478,3.582525e+06,1.143757,0.0,161.001608,1.362076,75.153726,2025.645693,6.326324,15.741386,5.397211e+04,10.366392,2006.145349
std,12638.029586,5820.194332,0.176527,782.139774,27.616906,12635.672543,6.677912e+05,1.256407,0.0,380.614313,6.455162,23.797627,0.478303,3.522899,8.737868,5.395654e+04,7.204957,11.249357
min,11110.000000,10100.000000,1.000000,0.000000,0.000000,11110.000000,2.000002e+06,0.000000,0.0,1.000000,0.000000,10.938000,2025.000000,1.000000,1.000000,8.000000e+02,-1.000000,1961.000000
25%,27230.000000,10400.000000,1.000000,355.000000,0.000000,27230.000000,3.167057e+06,0.000000,0.0,24.000000,0.000000,59.850000,2025.000000,3.000000,8.000000,2.200000e+04,5.000000,1997.000000
50%,41210.000000,10800.000000,1.000000,694.000000,0.000000,41210.000000,3.266040e+06,1.000000,0.0,59.000000,0.000000,78.167600,2026.000000,6.000000,16.000000,4.000000e+04,9.000000,2005.000000
75%,44131.000000,12400.000000,1.000000,1064.000000,1.000000,44131.000000,4.235418e+06,1.000000,0.0,142.000000,0.000000,84.960000,2026.000000,10.000000,23.000000,6.750000e+04,14.000000,2017.000000
max,52800.000000,47023.000000,7.000000,7375.000000,2837.000000,52800.000000,4.865212e+06,27.000000,0.0,8908.000000,154.000000,291.336000,2026.000000,12.000000,31.000000,2.500000e+06,82.000000,2026.000000


## 다음 단계

`02_eda_feature_engineering.ipynb`를 열어서 이어가면 됩니다. 그 노트북 맨 위에서 `data/processed/apt_trade_raw.parquet`와 `data/processed/sigungu_map.csv`를 로드하는 것부터 시작합니다.